In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import time
import optuna

from numba import njit

In [2]:
# ====================== PRE-COMPUTE z_t ======================
def compute_z(taker_buy_quote, quote_volume, num_trades):
    Q = np.array(quote_volume, dtype=np.float64)
    Bq = np.array(taker_buy_quote, dtype=np.float64)
    N = np.array(num_trades, dtype=np.float64)

    R = np.where(Q > 0, Bq / Q, 0.5)
    z = np.where((N > 0) & (Q > 0),
                 2 * (R - 0.5) * np.sqrt(N),
                 0.0)
    return z


# ====================== JITTED SIMULATION (exact PDF rules) ======================
@njit(fastmath=True, cache=True)
def simulate(closes: np.ndarray, f_target: np.ndarray, min_reb: float,
             c: float, initial_capital: float):
    n = len(closes)
    equity = np.empty(n, dtype=np.float64)
    equity[0] = initial_capital
    V = initial_capital
    f_prev = 0.0
    trades = 0

    for k in range(1, n):
        # Step 1: mark-to-market
        ret = closes[k] / closes[k - 1]
        V_minus = V * ret

        # Step 2-3: target + min-rebalance filter
        f_targ = f_target[k]
        if abs(f_targ - f_prev) > min_reb:
            f_k = f_targ
            trade_flag = True
        else:
            f_k = f_prev
            trade_flag = False

        # Step 4-5: apply cost
        tc = c * abs(f_k - f_prev) * V_minus if trade_flag else 0.0
        V = V_minus - tc

        if trade_flag and abs(f_k - f_prev) > 1e-8:
            trades += 1

        equity[k] = V
        f_prev = f_k

    return equity, trades


def compute_metrics(equity: np.ndarray, initial_capital: float, n: int):
    final_pnl = equity[-1] - initial_capital
    if n < 2:
        return {'final_pnl': final_pnl, 'sharpe': 0.0, 'ann_sharpe': 0.0, 'max_dd_pct': 0.0}

    returns = np.diff(equity) / equity[:-1]
    sharpe = 0.0
    ann_sharpe = 0.0
    if len(returns) > 30 and np.std(returns) > 1e-8:
        sharpe = np.mean(returns) / np.std(returns)
        ann_sharpe = sharpe * np.sqrt(365.25)

    cummax = np.maximum.accumulate(equity)
    dd = (equity - cummax) / cummax
    max_dd_pct = -dd.min() * 100 if dd.min() < 0 else 0.0

    return {
        'final_pnl': final_pnl,
        'sharpe': sharpe,
        'ann_sharpe': ann_sharpe,
        'max_dd_pct': max_dd_pct
    }

In [ ]:
csv_path = "data/BTCUSDT-1d-2023-01-01_2026-02-28.csv"   # UPDATE TO YOUR FILE

print("Loading CSV...")
df = pd.read_csv(csv_path)
df['datetime'] = pd.to_datetime(df["OpenTimestamp[ms]"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)
n = len(df)
print(f"Data loaded: {n:,} minutes (~{n/(60*24):.1f} days)")

# Exact columns
closes = df['Close'].values.astype(np.float64)
quote_vol = df['QuoteAssetVolume'].values
num_trades = df['NumberOfTrades'].values
taker_buy_quote = df['TakerBuyQuoteVolume'].values

z = compute_z(taker_buy_quote, quote_vol, num_trades)

initial_capital = 1_000_000.0
c = 0.001
alpha = 0.006
k = 3.0
min_reb = 0.01

# ====================== DEFAULT BACKTEST ======================
print(f"\nRunning default (α={alpha}, k={k}, min_reb={min_reb})...")
p_def = pd.Series(z).ewm(alpha=alpha, adjust=False).mean().values
f_target_def = np.clip(p_def / k, 0.0, 1.0)
equity_before, trades_before = simulate(closes, f_target_def, min_reb, 0.0, initial_capital)
equity_after, trades_after = simulate(closes, f_target_def, min_reb, c, initial_capital)
metrics_before = compute_metrics(equity_before, initial_capital, n)
metrics_after = compute_metrics(equity_after, initial_capital, n)

print("\n" + "="*80)
print(f"BACKTEST RESULTS (α={alpha}, k={k}, min_reb={min_reb})")
print("="*80)
print(f"Final PnL BEFORE         : ${metrics_before['final_pnl']:,.2f}")
print(f"Final PnL AFTER          : ${metrics_after['final_pnl']:,.2f}")
print(f"Sharpe Ratio BEFORE      : {metrics_before['sharpe']:.4f}")
print(f"Sharpe Ratio AFTER       : {metrics_after['sharpe']:.4f}")
print(f"Annualized Sharpe BEFORE : {metrics_before['ann_sharpe']:.2f}")
print(f"Annualized Sharpe AFTER  : {metrics_after['ann_sharpe']:.2f}")
print(f"Max Drawdown BEFORE      : {metrics_before['max_dd_pct']:.2f}%")
print(f"Max Drawdown AFTER       : {metrics_after['max_dd_pct']:.2f}%")
print(f"Total trades BEFORE      : {trades_before:,}")
print(f"Total trades AFTER       : {trades_after:,}")
print("="*80)

equity_df = pd.DataFrame({'datetime': df['datetime'], 'pnl_before': equity_before - initial_capital, 'pnl_after': equity_after - initial_capital})
equity_df.to_csv('equity_curve.csv', index=False)
print("Saved 'equity_curve.csv' for the Flask dashboard.")

# Cumulative PnL plot (static)
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before - initial_capital, label='Before 0.1% cost', color='royalblue')
plt.plot(df['datetime'], equity_after - initial_capital, label='After 0.1% cost', color='crimson')
plt.title(f'Strategy Backtest (α={alpha}, k={k}, min_reb={min_reb})', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
n_trials = 1_000

def objective(trial, closes, z, initial_capital, n, c=0.001):
    alpha = trial.suggest_float('alpha', 0.001, 0.3)
    k = trial.suggest_float('k', 1.5, 5.0)
    min_reb = trial.suggest_float('min_reb', 0.005, 0.05)

    # EWMA smoothing (still pandas – very fast)
    p = pd.Series(z).ewm(alpha=alpha, adjust=False).mean().values
    f_target = np.clip(p / k, 0.0, 1.0)

    # JITted core simulation
    equity, trades = simulate(closes, f_target, min_reb, c, initial_capital)
    metrics = compute_metrics(equity, initial_capital, n)
    return metrics['ann_sharpe']


# ====================== OPTUNA OPTIMIZATION (now blazing fast) ======================
print(f"\nStarting Optuna ({n_trials} trials) — tuning α, k, min_reb ...")
start_time = time.time()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(lambda trial: objective(trial, closes, z, initial_capital, n, c),
               n_trials=n_trials, show_progress_bar=True)

best = study.best_params
best_sharpe = study.best_value
print(f"\nOPTUNA BEST: α={best['alpha']:.3f}, k={best['k']:.2f}, min_reb={best['min_reb']:.3f} → Sharpe = {best_sharpe:.4f}")
print(f"Finished in {time.time() - start_time:.1f}s")

# ====================== RUN BEST & SAVE ======================
print("\nRunning best parameters...")
p_best = pd.Series(z).ewm(alpha=best['alpha'], adjust=False).mean().values
f_target_best = np.clip(p_best / best['k'], 0.0, 1.0)
equity_best, trades_best = simulate(closes, f_target_best, best['min_reb'], c, initial_capital)
metrics_best = compute_metrics(equity_best, initial_capital, n)

equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl_before': equity_best - initial_capital,
    'pnl_after':  equity_best - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("✅ Saved 'equity_curve.csv' (BEST parameters)")

print("\n" + "="*80)
print(f"BEST RESULTS (α={best['alpha']:.3f}, k={best['k']:.2f}, min_reb={best['min_reb']:.3f})")
print("="*80)
print(f"Final PnL         : ${metrics_best['final_pnl']:,.2f}")
print(f"Sharpe Ratio      : {metrics_best['sharpe']:.4f}")
print(f"Ann. Sharpe       : {metrics_best['ann_sharpe']:.2f}")
print(f"Max DD            : {metrics_best['max_dd_pct']:.2f}%")
print(f"Total trades      : {trades_best:,}")
print("="*80)

plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_best - initial_capital, color='royalblue', linewidth=1.5)
plt.title(f'BEST STBAI Strategy (α={best["alpha"]:.3f}, k={best["k"]:.2f}, min_reb={best["min_reb"]:.3f})', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()